In [1]:
import os
import rootutils
import pandas as pd
import random
import numpy as np

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
tqdm.pandas()

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

In [2]:
from src.cif_utils import cif_from_file

In [3]:
meta_csv = "data_cod/cod_bradley_merged.csv"
cifs_dir = "cifs"

In [4]:
meta_df = pd.read_csv(meta_csv).drop_duplicates().sort_values(by="id")
meta_df["cif_path"] = cifs_dir + "/" + meta_df["id"].astype(str) + ".cif"

---
## Coordinational Numbers:

In [5]:
from pymatgen.core import Structure
import numpy as np
import pandas as pd
from collections import defaultdict

In [10]:
def is_interior(site, frac_margin):
    f = site.frac_coords
    return np.all(f > frac_margin) and np.all(f < 1 - frac_margin)

def compute_avg_cn_df(cif_path: str, raw_cutoffs: dict) -> pd.DataFrame:
    """
    Returns a DataFrame with columns:
      central, neighbor, avg_cn, n_sites
    """
    # Normalize cutoffs to unordered frozenset keys
    cutoffs = {frozenset((e1, e2)): cutoff
               for (e1, e2), cutoff in raw_cutoffs.items()}
    
    struct = Structure.from_file(cif_path)
    elems = sorted({site.specie.symbol for site in struct})
    
    max_cut = max(cutoffs.values())
    frac_margin = max_cut / min(struct.lattice.abc)
    
    coord_numbers = defaultdict(lambda: defaultdict(list))
    
    for site in struct:
        if not is_interior(site, frac_margin):
            continue
        central = site.specie.symbol
        for pair, cutoff in cutoffs.items():
            if central not in pair:
                continue
            partner = next(iter(pair - {central}))
            neighs = struct.get_neighbors(site, cutoff)
            cnt = sum(1 for n in neighs if n.specie.symbol == partner)
            coord_numbers[central][partner].append(cnt)
    
    # Build DataFrame of observed stats
    rows = []
    for central in elems:
        for partner in elems:
            if central == partner:
                continue
            counts = coord_numbers[central].get(partner, [])
            avg_cn = float(np.mean(counts)) if counts else 0.0
            n_sites = len(counts)
            rows.append({
                "central": central,
                "neighbor": partner,
                "avg_cn": avg_cn,
                "n_sites": n_sites
            })
    df = pd.DataFrame(rows)
    return df


def get_fixed_length_descriptor(cif_path: str, raw_cutoffs: dict) -> pd.Series:
    """
    Returns a pandas Series of length = number of raw_cutoffs * 2 (central->neighbor),
    indexed by strings "E1–E2", in sorted order.
    """
    # 1) Compute the avg_cn DataFrame
    df = compute_avg_cn_df(cif_path, raw_cutoffs)
    
    #2) Build the list of all feature names in a fixed order:
    #    we use exactly the ordered keys of raw_cutoffs (both directions)
    features = []
    for (e1, e2) in sorted(raw_cutoffs.keys()):
        features.append(f"{e1}–{e2}")
        features.append(f"{e2}–{e1}")
    
    # 3) Map each feature onto its avg_cn (or 0.0 if missing)
    avg_cn_map = {
        f"{row.central}–{row.neighbor}": row.avg_cn
        for _, row in df.iterrows()
    }
    descriptor = {feat: avg_cn_map.get(feat, 0.0) for feat in features}
    
    # 4) Return as Series
    return pd.Series(descriptor, name="avg_cn_descriptor")


In [11]:
cutoffs = {
    ("O", "H"): 1.27,
    ("O", "C"): 1.72,
    ("C", "H"): 1.37,
    ("C", "N"): 1.60
}

# cutoffs = {
#     ("C", "H"): 1.20,   # C–H bond length ≈ 1.09 Å + buffer
#     ("C", "C"): 1.70,   # C–C single bond ≈ 1.54 Å + buffer
#     ("C", "N"): 1.60,   # C–N single bond ≈ 1.47 Å + buffer
#     ("C", "O"): 1.60,   # C–O single bond ≈ 1.43 Å + buffer
#     ("C", "S"): 1.90,   # C–S single bond ≈ 1.82 Å + buffer
#     ("C", "Cl"): 1.90,  # C–Cl bond length ≈ 1.76 Å + buffer
#     ("C", "Br"): 2.00,  # C–Br bond length ≈ 1.94 Å + buffer
#     ("C", "I"): 2.20,   # C–I bond length ≈ 2.14 Å + buffer
#     ("N", "H"): 1.10,   # N–H bond length ≈ 1.01 Å + buffer
#     ("N", "O"): 1.50,   # N–O single bond ≈ 1.45 Å + buffer
#     ("N", "N"): 1.50,   # N–N single bond ≈ 1.45 Å + buffer
#     ("O", "H"): 1.10,   # O–H bond length ≈ 0.97 Å + buffer
#     ("O", "O"): 1.50,   # O–O single bond ≈ 1.45 Å + buffer
#     ("S", "H"): 1.40,   # S–H bond length ≈ 1.34 Å + buffer
#     ("S", "O"): 1.60,   # S–O single bond ≈ 1.48 Å + buffer
#     ("Cl", "H"): 1.30,  # H–Cl bond length ≈ 1.27 Å + buffer
#     ("Br", "H"): 1.50,  # H–Br bond length ≈ 1.41 Å + buffer
#     ("I", "H"): 1.70,   # H–I bond length ≈ 1.61 Å + buffer
# }


In [12]:

error_cifs = [] 
for i, cif_path in tqdm(enumerate(meta_df["cif_path"]), total=len(meta_df), desc="Processing CIFs"):
    try:
        coord_numbs = get_fixed_length_descriptor(cif_path, cutoffs)
    except Exception as e:
        print(f"Error: {e}")
        error_cifs.append(cif_path)
        
    if i >= 0:
        break


Processing CIFs:   0%|          | 0/13528 [00:00<?, ?it/s]

In [13]:
coord_numbs

C–H    1.363636
H–C    0.857143
C–N    0.000000
N–C    0.000000
O–C    1.000000
C–O    0.363636
O–H    1.000000
H–O    0.142857
Name: avg_cn_descriptor, dtype: float64